# 🤖 Extractor de Criterios RIASEC con Gemini

Este notebook procesa los archivos markdown de cada carrera y utiliza Gemini, guiado por `prompt.md`, para extraer criterios RIASEC.

## 🎯 Objetivo

Por cada perfil de carrera en formato markdown (`[carrera_id].md`):
1. **ETAPA 1 · PERFIL**: extraer 1 criterio por dimensión desde los perfiles de ingreso y egreso
2. **ETAPA 2 · INTERNET**: completar solo las dimensiones obligatorias faltantes (R, I, S, E, C) con Google Search
3. **ETAPA 3 · CONTINGENCIA**: construir con la estructura estándar las que sigan faltando
4. Generar `resultado_criterios.xlsx` con 3 hojas: `criterios`, `fuentes`, `cobertura_riasec`

## 📋 Regla de cobertura

Por carrera: **exactamente 1** criterio R, I, S, E, C y **0 o 1** criterio A (nunca por INTERNET ni CONTINGENCIA).
Todos los registros con `type_id = 1`.

## 📦 Dependencias e Imports

In [ ]:
import json
import re
import time
from datetime import date
from pathlib import Path
from typing import Optional

import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel
from tqdm import tqdm

from lib.set_envs import GEMINI_API_KEY

## ⚙️ Configuración de Gemini

In [ ]:
MODEL_NAME = "gemini-2.5-pro"

client = genai.Client(api_key=GEMINI_API_KEY)

print(f"✅ Gemini configurado correctamente")
print(f"🤖 Modelo: {MODEL_NAME}")

## 📂 Configuración de Rutas

In [ ]:
PROMPT_PATH = Path("../assets/prompts/prompt.md")
MARKDOWN_DIR = Path("../assets/layouts/markdown")
OUTPUT_DIR = Path("../assets/media/extraction_result")
EXCEL_PATH = OUTPUT_DIR / "resultado_criterios.xlsx"
RAW_DIR = OUTPUT_DIR / "raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)
FECHA_CONSULTA = date.today()

print(f"📄 Prompt: {PROMPT_PATH}")
print(f"📁 Markdown dir: {MARKDOWN_DIR}")
print(f"📊 Excel output: {EXCEL_PATH}")

## 🧬 Modelo RIASEC

Constantes de la sección 3 y de la ETAPA 3 del prompt. La contingencia se construye en Python con las estructuras estándar, así el texto es determinista y nunca se genera para A.

In [ ]:
RIASEC_ID = {"R": 1, "I": 2, "A": 3, "S": 4, "E": 5, "C": 6}
OBLIGATORIAS = ["R", "I", "S", "E", "C"]
DIMENSIONES = ["R", "I", "A", "S", "E", "C"]
TYPE_ID = 1 

CONTINGENCIA = {
    "R": ("Aplicar procedimientos técnicos de {area}", "Procedimientos técnicos de {area}"),
    "I": ("Investigar temas de {area}", "Investigación en {area}"),
    "S": ("Orientar sobre temas de {area}", "Orientación en {area}"),
    "E": ("Emprender proyectos de {area}", "Emprendimiento en {area}"),
    "C": ("Organizar información de {area}", "Organización de información de {area}"),
}

## 📖 Cargar Prompt Template

In [ ]:
with open(PROMPT_PATH, 'r', encoding='utf-8') as f:
    prompt_template = f.read()

print(f"✅ Prompt cargado ({len(prompt_template)} caracteres)")
print(f"\n--- Preview del prompt ---")
print(prompt_template[:300] + "..." if len(prompt_template) > 300 else prompt_template)

## 🧾 Esquemas de Respuesta

El prompt pide un Excel completo, pero aquí se procesa **una carrera por llamada** y Python arma las 3 hojas.
Por eso cada etapa añade una instrucción que reemplaza la sección 9 (formato de salida) por JSON.

El esquema de la ETAPA 1 tiene un campo por dimensión, lo que impide de forma estructural devolver 2 criterios de la misma dimensión.

In [ ]:
class Criterio(BaseModel):
    desc_criterio: str
    resumen_criterio: str


class Etapa1(BaseModel):
    nombre_carrera: str
    area_disciplinar: str
    R: Optional[Criterio] = None
    I: Optional[Criterio] = None
    A: Optional[Criterio] = None
    S: Optional[Criterio] = None
    E: Optional[Criterio] = None
    C: Optional[Criterio] = None


INSTRUCCION_ETAPA1 = """
Ejecuta ÚNICAMENTE la ETAPA 1 (PERFIL) del prompt para la carrera del documento adjunto.
- carrera_id = {carrera_id} (archivo original {carrera_id}.pdf, convertido a markdown).
- Analiza "Perfil de ingreso" Y "Perfil de egreso" y selecciona la competencia más representativa por dimensión.
- Si una dimensión no tiene evidencia en ninguno de los dos perfiles, devuélvela como null. NO inventes.
- Aplica las reglas de normalización (sección 7) y de campos de salida (sección 8).
- Ignora la sección 9: NO generes Excel; responde solo con el JSON solicitado.

--- DOCUMENTO ---
{markdown}
"""

INSTRUCCION_ETAPA2 = """
Ejecuta ÚNICAMENTE la ETAPA 2 (INTERNET) del prompt.
Carrera: {nombre_carrera} (carrera_id = {carrera_id}).
Dimensiones obligatorias faltantes tras revisar ambos perfiles: {faltantes}.

Para CADA dimensión faltante busca con Google Search 1 competencia respaldada por una fuente aceptable
(respeta la jerarquía de fuentes y las fuentes NO aceptables del prompt). NO busques la dimensión A.
Si no encuentras evidencia verificable, marca encontrada=false y explica en "nota" qué consultaste.

Responde SOLO con un bloque ```json con esta forma:
{{
  "<DIM>": {{
    "encontrada": true | false,
    "desc_criterio": "...", "resumen_criterio": "...",
    "titulo_fuente": "...", "institucion": "...", "url": "...",
    "evidencia_utilizada": "fragmento textual de la fuente",
    "nota": "relación con la carrera, p. ej. 'Programa equivalente: ... Perfil de egreso.'",
    "urls_consultadas": ["..."]
  }}
}}
"""

print("✅ Esquemas e instrucciones definidos")

## 🔍 Listar Archivos Markdown

In [ ]:
markdown_files = sorted(MARKDOWN_DIR.glob("*.md"), key=lambda x: int(x.stem) if x.stem.isdigit() else 0)

print(f"📊 Total de archivos markdown: {len(markdown_files)}")
print(f"\nPrimeros 5 archivos:")
for f in markdown_files[:5]:
    print(f"  - {f.name}")

## 🤖 Funciones de Procesamiento

In [ ]:
def call_with_retry(fn, retries: int = 3, wait: int = 10):
    for attempt in range(1, retries + 1):
        try:
            return fn()
        except Exception:
            if attempt == retries:
                raise
            time.sleep(wait * attempt)


def etapa1_perfil(carrera_id: int, markdown_content: str) -> Etapa1:
    response = call_with_retry(lambda: client.models.generate_content(
        model=MODEL_NAME,
        contents=INSTRUCCION_ETAPA1.format(carrera_id=carrera_id, markdown=markdown_content),
        config=types.GenerateContentConfig(
            system_instruction=prompt_template,
            response_mime_type="application/json",
            response_schema=Etapa1,
            temperature=0.2,
        ),
    ))
    return response.parsed


def etapa2_internet(carrera_id: int, nombre_carrera: str, faltantes: list[str]) -> dict:
    response = call_with_retry(lambda: client.models.generate_content(
        model=MODEL_NAME,
        contents=INSTRUCCION_ETAPA2.format(
            carrera_id=carrera_id, nombre_carrera=nombre_carrera, faltantes=", ".join(faltantes)
        ),
        config=types.GenerateContentConfig(
            system_instruction=prompt_template,
            tools=[types.Tool(google_search=types.GoogleSearch())],
            temperature=0.2,
        ),
    ))
    match = re.search(r"```json\s*(\{.*\})\s*```", response.text, re.DOTALL)
    data = json.loads(match.group(1) if match else response.text)
    return {dim: data[dim] for dim in faltantes if dim in data}


def process_career_markdown(file_path: Path) -> dict:
    carrera_id = int(file_path.stem)
    cache_path = RAW_DIR / f"{carrera_id}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))

    markdown_content = file_path.read_text(encoding="utf-8")

    perfil = etapa1_perfil(carrera_id, markdown_content)
    faltantes = [dim for dim in OBLIGATORIAS if getattr(perfil, dim) is None]
    internet = etapa2_internet(carrera_id, perfil.nombre_carrera, faltantes) if faltantes else {}

    result = {
        "carrera_id": carrera_id,
        "perfil": perfil.model_dump(),
        "internet": internet,
    }
    cache_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result

print("✅ Funciones de procesamiento definidas")

## 🚀 Procesamiento en Lote

Este proceso puede tardar varios minutos dependiendo de la cantidad de archivos y la velocidad de la API.
Las carreras ya procesadas se leen de `raw/`, así que si se interrumpe basta con volver a ejecutar la celda.

Los archivos sin identificador numérico **no se procesan** y se registran como `NO PROCESADO` (sección 4 del prompt).

In [ ]:
results = []
errors = []
no_procesados = [f for f in MARKDOWN_DIR.glob("*.md") if not f.stem.isdigit()]

print(f"🚀 Iniciando procesamiento de {len(markdown_files)} archivos...\n")

for file_path in tqdm(markdown_files, desc="Procesando carreras"):
    if not file_path.stem.isdigit():
        continue
    try:
        results.append(process_career_markdown(file_path))
    except Exception as e:
        errors.append({"archivo": file_path.name, "error": str(e)})

    time.sleep(1)

print(f"\n✅ Procesamiento completado")
print(f"   Exitosos: {len(results)}")
print(f"   Errores: {len(errors)}")
print(f"   No procesados (sin carrera_id): {len(no_procesados)}")

## 📊 Estructurar Datos para Excel

Aquí se aplica la jerarquía **PERFIL → INTERNET → CONTINGENCIA** y se construyen las 3 hojas:
- `fuente_id` es un consecutivo global, solo para criterios INTERNET
- CONTINGENCIA usa la estructura estándar con el área de la carrera
- `observaciones` documenta los complementos externos y las contingencias

In [ ]:
criterios_list = []
fuentes_list = []
cobertura_list = []

for result in sorted(results, key=lambda r: r["carrera_id"]):
    carrera_id = result["carrera_id"]
    perfil = result["perfil"]
    internet = result["internet"]
    area = perfil["area_disciplinar"]

    origen_por_dim = {}
    contingencias = []
    complementos = []
    obs_contingencia = []

    for dim in DIMENSIONES:
        fuente_id = None
        criterio = perfil.get(dim)

        if criterio:
            origen = "PERFIL"
        elif dim == "A":
            origen_por_dim[dim] = "OPCIONAL NO ENCONTRADA"
            continue
        elif internet.get(dim, {}).get("encontrada"):
            origen = "INTERNET"
            criterio = internet[dim]
            fuente_id = len(fuentes_list) + 1
            fuentes_list.append({
                "fuente_id": fuente_id,
                "carrera_id": carrera_id,
                "riasec_id": RIASEC_ID[dim],
                "type_id": TYPE_ID,
                "titulo_fuente": criterio["titulo_fuente"],
                "institucion": criterio["institucion"],
                "url": criterio["url"],
                "fecha_consulta": FECHA_CONSULTA,
                "evidencia_utilizada": criterio["evidencia_utilizada"],
            })
            complementos.append(f"{dim}: {criterio['nota']}")
        else:
            origen = "CONTINGENCIA"
            desc, resumen = CONTINGENCIA[dim]
            criterio = {"desc_criterio": desc.format(area=area), "resumen_criterio": resumen.format(area=area)}
            contingencias.append(dim)
            intento = internet.get(dim, {})
            consultas = " ; ".join(intento.get("urls_consultadas", []))
            obs_contingencia.append(
                f"{dim} completada mediante CONTINGENCIA. {intento.get('nota', '')} "
                f"Criterio construido: {criterio['desc_criterio']}."
                + (f" Consultas: {consultas} ." if consultas else "")
            )

        origen_por_dim[dim] = origen
        criterios_list.append({
            "carrera_id": carrera_id,
            "type_id": TYPE_ID,
            "riasec_id": RIASEC_ID[dim],
            "desc_criterio": criterio["desc_criterio"],
            "resumen_criterio": criterio["resumen_criterio"],
            "origen": origen,
            "fuente_id": fuente_id,
        })

    observaciones = f"Se revisaron los perfiles de ingreso y egreso de {carrera_id}.pdf."
    if complementos:
        observaciones += " Complemento externo: " + " ".join(complementos)
    if obs_contingencia:
        observaciones += " " + " ".join(obs_contingencia)

    cobertura_list.append({
        "carrera_id": carrera_id,
        "nombre_carrera": perfil["nombre_carrera"],
        **origen_por_dim,
        "dimensiones_completadas_por_contingencia": ", ".join(contingencias) or None,
        "dimensiones_faltantes": None,
        "estado": "COMPLETO CON CONTINGENCIA" if contingencias else "COMPLETO CON EVIDENCIA",
        "observaciones": re.sub(r"\s+", " ", observaciones).strip(),
    })

for f in no_procesados:
    cobertura_list.append({
        "carrera_id": None,
        "nombre_carrera": None,
        "estado": "NO PROCESADO",
        "observaciones": f"{f.name} no tiene identificador numérico reconocible.",
    })

df_criterios = pd.DataFrame(criterios_list, columns=[
    "carrera_id", "type_id", "riasec_id", "desc_criterio", "resumen_criterio", "origen", "fuente_id",
]).astype({"fuente_id": "Int64"})
df_fuentes = pd.DataFrame(fuentes_list, columns=[
    "fuente_id", "carrera_id", "riasec_id", "type_id", "titulo_fuente",
    "institucion", "url", "fecha_consulta", "evidencia_utilizada",
])
df_cobertura = pd.DataFrame(cobertura_list, columns=[
    "carrera_id", "nombre_carrera", *DIMENSIONES, "dimensiones_completadas_por_contingencia",
    "dimensiones_faltantes", "estado", "observaciones",
])

print(f"\n📊 Datos estructurados:")
print(f"   Criterios: {len(df_criterios)} registros")
print(f"   Fuentes: {len(df_fuentes)} registros")
print(f"   Cobertura: {len(df_cobertura)} carreras")

## ✅ Control de Calidad

Checklist de la sección 11 del prompt antes de escribir el Excel.

In [ ]:
conteo = df_criterios.groupby(["carrera_id", "riasec_id"]).size().unstack(fill_value=0)
obligatorias_ids = [RIASEC_ID[d] for d in OBLIGATORIAS]

assert (conteo.reindex(columns=obligatorias_ids, fill_value=0) == 1).all().all(), "Hay carreras sin exactamente 1 R, I, S, E, C"
assert (conteo.get(RIASEC_ID["A"], pd.Series(0)) <= 1).all(), "Hay carreras con más de 1 criterio A"
assert (df_criterios["type_id"] == 1).all(), "type_id debe ser siempre 1"
assert not ((df_criterios["riasec_id"] == RIASEC_ID["A"]) & (df_criterios["origen"] != "PERFIL")).any(), "A solo puede venir de PERFIL"

internet = df_criterios[df_criterios["origen"] == "INTERNET"]
assert internet["fuente_id"].notna().all(), "Todo criterio INTERNET necesita fuente_id"
assert df_criterios.loc[df_criterios["origen"] != "INTERNET", "fuente_id"].isna().all(), "PERFIL/CONTINGENCIA sin fuente_id"
assert set(internet["fuente_id"]) == set(df_fuentes["fuente_id"]), "criterios.fuente_id y fuentes.fuente_id no coinciden"
assert (df_criterios["resumen_criterio"].str.len() < df_criterios["desc_criterio"].str.len()).all(), "resumen_criterio debe ser más corto"

print("✅ Control de calidad superado")
print(df_criterios["origen"].value_counts().to_string())

## 💾 Generar Excel

In [ ]:
with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
    df_criterios.to_excel(writer, sheet_name='criterios', index=False)
    df_fuentes.to_excel(writer, sheet_name='fuentes', index=False)
    df_cobertura.to_excel(writer, sheet_name='cobertura_riasec', index=False)

print(f"\n✅ Excel generado: {EXCEL_PATH}")
print(f"📄 Hojas:")
print(f"   - criterios ({len(df_criterios)} registros)")
print(f"   - fuentes ({len(df_fuentes)} registros)")
print(f"   - cobertura_riasec ({len(df_cobertura)} registros)")

## 📊 Visualizar Muestra de Resultados

In [ ]:
print("\n=== MUESTRA CRITERIOS ===")
print(df_criterios.head(5))

print("\n=== CARRERAS CON CONTINGENCIA ===")
print(df_cobertura[df_cobertura["estado"] == "COMPLETO CON CONTINGENCIA"][["carrera_id", "nombre_carrera", "dimensiones_completadas_por_contingencia"]])

# Mostrar errores si hay
if errors:
    print("\n⚠️  ERRORES ENCONTRADOS:")
    for e in errors:
        print(f"   {e['archivo']}: {e['error']}")